In [55]:
import pandas as pd
df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')
df.head(2)

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin


In [56]:
# Tokenized
def tokenize(text):
  text = text.lower()
  text = text.replace('\n', ' ')
  return text.split()

In [57]:
# Vocab
vocab = {'<LINK>':0}

In [58]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])
  merged_tokens = tokenized_question + tokenized_answer
  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [59]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [60]:
vocab

{'<LINK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france?': 6,
 'paris': 7,
 'germany?': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 "'to": 12,
 'kill': 13,
 'a': 14,
 "mockingbird'?": 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system?': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius?': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa?': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64?': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold?': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end?': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'world?': 52,
 'nile': 53,
 'japan?': 54,
 'tokyo': 55,
 'developed': 56,
 'theory': 57,
 'relativity?': 58,
 'albert-einstein': 59,
 'freezing': 60,
 'fahrenheit?': 61,
 '32': 62,
 'known': 63,
 'as': 64,
 'red': 65,
 'planet?': 66,
 'mars': 67,
 'author': 68,
 "'1984'?": 

In [61]:
# convert words to numerical indices
def text_to_indices(text, vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<LINK>'])
  return indexed_text

In [62]:
text_to_indices('what is campus x', vocab)

[1, 2, 0, 0]

In [63]:
import torch
from torch.utils.data import Dataset, DataLoader

In [64]:
class QADataset(Dataset):
  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [65]:
dataset = QADataset(df, vocab)

In [66]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [86]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[ 42, 224, 121, 225, 226,  19,  14, 227, 228]]) tensor([229])
tensor([[  1,   2,   3,  17, 118,  85,  86]]) tensor([119])
tensor([[ 42, 261, 262, 121, 263, 264]]) tensor([265])
tensor([[ 42, 266,   2, 267,  85, 268, 269]]) tensor([270])
tensor([[ 42, 181,   2,  63,  39, 182, 183, 149, 184, 185]]) tensor([186])
tensor([[ 10, 146,   3, 147, 178,   5,   3,  72, 179]]) tensor([180])
tensor([[  1,   2,   3,  37,  38,  39, 168]]) tensor([169])
tensor([[ 42,  18,   2,   3, 293, 149,   3, 294]]) tensor([212])
tensor([[  1,   2,   3,  33,  34,   5, 256]]) tensor([257])
tensor([[ 42, 142,   2, 143,  39, 182, 280]]) tensor([281])
tensor([[ 42, 302, 303, 121, 304, 165, 305, 306]]) tensor([307])
tensor([[42, 18,  2, 63, 64,  3, 65, 66]]) tensor([67])
tensor([[ 42, 129,   2,  63,  64,   3, 130, 131]]) tensor([132])
tensor([[  1,   2,   3,   4,   5, 247, 248]]) tensor([249])
tensor([[10, 56,  3, 57,  5, 58]]) tensor([59])
tensor([[  1,   2,   3, 147, 120,  85,   3, 289, 290]]) tensor([125])
t

In [68]:
import torch.nn as nn

In [124]:
class SimpleRNN(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    print(final.shape)
    output = self.fc(final[-1])
    return output

In [101]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [116]:
learning_rate = 0.001
epoch = 20

In [117]:
mmodel = SimpleRNN(len(vocab))

In [118]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mmodel.parameters(), lr=learning_rate)

In [119]:
# training loop

for epoch_idx in range(epoch):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = mmodel(question)

    # Ensure prediction is [batch_size, vocab_size] and target is [batch_size]
    loss = criterion(output.view(-1, len(vocab)), answer.view(-1))

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch_idx+1}, Loss: {total_loss:.4f}")

Epoch: 1, Loss: 528.6530
Epoch: 2, Loss: 462.0636
Epoch: 3, Loss: 383.3344
Epoch: 4, Loss: 320.0419
Epoch: 5, Loss: 268.4808
Epoch: 6, Loss: 221.2540
Epoch: 7, Loss: 178.5389
Epoch: 8, Loss: 139.4572
Epoch: 9, Loss: 107.0973
Epoch: 10, Loss: 82.0228
Epoch: 11, Loss: 62.9876
Epoch: 12, Loss: 48.9226
Epoch: 13, Loss: 38.4225
Epoch: 14, Loss: 30.9268
Epoch: 15, Loss: 25.5563
Epoch: 16, Loss: 21.0516
Epoch: 17, Loss: 17.6763
Epoch: 18, Loss: 15.0424
Epoch: 19, Loss: 13.1202
Epoch: 20, Loss: 11.4160


In [120]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [122]:
predict(mmodel, "What is the largest planet in our solar system?")

jupiter


In [123]:
list(vocab.keys())[7]

'paris'